# Divvy Bikeshare Data Lake Project

## Overview
This project implements a Lakehouse architecture using Azure Databricks and Delta Lake.

## Business Requirements
- Analyze ride duration
- Analyze usage by time
- Analyze usage by station
- Analyze usage by rider age
- Analyze revenue

In [0]:
df_trip_raw = spark.read.format("csv") \
    .option("inferSchema", "true") \
    .option("header", "false") \
    .option("sep", ",") \
    .load("dbfs:/FileStore/tables/trips.csv")

df_trip_raw.show(3)

+----------------+-------------+-------------------+-------------------+------------+------------+-----+
|             _c0|          _c1|                _c2|                _c3|         _c4|         _c5|  _c6|
+----------------+-------------+-------------------+-------------------+------------+------------+-----+
|89E7AA6C29227EFF| classic_bike|2021-02-12 16:14:56|2021-02-12 16:21:43|         525|         660|71934|
|0FEFDE2603568365| classic_bike|2021-02-14 17:52:38|2021-02-14 18:12:09|         525|       16806|47854|
|E6159D746B2DBB91|electric_bike|2021-02-09 19:10:18|2021-02-09 19:19:10|KA1503000012|TA1305000029|70870|
+----------------+-------------+-------------------+-------------------+------------+------------+-----+
only showing top 3 rows


In [0]:
df_trip = df_trip_raw.toDF(
    "ride_id",
    "rideable_type",
    "started_at",
    "ended_at",
    "start_station_id",
    "end_station_id",
    "rider_id"
)

df_trip.show(3)

+----------------+-------------+-------------------+-------------------+----------------+--------------+--------+
|         ride_id|rideable_type|         started_at|           ended_at|start_station_id|end_station_id|rider_id|
+----------------+-------------+-------------------+-------------------+----------------+--------------+--------+
|89E7AA6C29227EFF| classic_bike|2021-02-12 16:14:56|2021-02-12 16:21:43|             525|           660|   71934|
|0FEFDE2603568365| classic_bike|2021-02-14 17:52:38|2021-02-14 18:12:09|             525|         16806|   47854|
|E6159D746B2DBB91|electric_bike|2021-02-09 19:10:18|2021-02-09 19:19:10|    KA1503000012|  TA1305000029|   70870|
+----------------+-------------+-------------------+-------------------+----------------+--------------+--------+
only showing top 3 rows


In [0]:
df_trip.write.format("delta") \
    .mode("overwrite") \
    .save("/delta/bronze/trip")


spark.read.format("delta") \
    .load("/delta/bronze/trip") \
    .show(3)

+----------------+-------------+-------------------+-------------------+----------------+--------------+--------+
|         ride_id|rideable_type|         started_at|           ended_at|start_station_id|end_station_id|rider_id|
+----------------+-------------+-------------------+-------------------+----------------+--------------+--------+
|493E7531A95A143E| classic_bike|2021-06-02 17:24:19|2021-06-02 17:51:49|           13053|  TA1307000150|    8563|
|A3B7CB97068AA221| classic_bike|2021-06-10 20:50:10|2021-06-10 21:20:42|           13053|  TA1307000150|   35607|
|E65F52226B594B89| classic_bike|2021-06-05 09:33:36|2021-06-05 09:36:34|           13001|  KA1503000015|   75919|
+----------------+-------------+-------------------+-------------------+----------------+--------------+--------+
only showing top 3 rows


In [0]:
# =========================
# PAYMENT
# =========================
df_payment_raw = spark.read.format("csv") \
    .option("inferSchema", "true") \
    .option("header", "false") \
    .option("sep", ",") \
    .load("dbfs:/FileStore/tables/payments.csv")

df_payment = df_payment_raw.toDF(
    "payment_id",
    "date",
    "amount",
    "account_number"
)

df_payment.write.format("delta").mode("overwrite").save("/delta/bronze/payment")

spark.read.format("delta").load("/delta/bronze/payment").show(3)


# =========================
# STATION
# =========================
df_station_raw = spark.read.format("csv") \
    .option("inferSchema", "true") \
    .option("header", "false") \
    .option("sep", ",") \
    .load("dbfs:/FileStore/tables/stations.csv")

df_station = df_station_raw.toDF(
    "station_id",
    "name",
    "latitude",
    "longitude"
)

df_station.write.format("delta").mode("overwrite").save("/delta/bronze/station")

spark.read.format("delta").load("/delta/bronze/station").show(3)


# =========================
# RIDER
# =========================
df_rider_raw = spark.read.format("csv") \
    .option("inferSchema", "true") \
    .option("header", "false") \
    .option("sep", ",") \
    .load("dbfs:/FileStore/tables/riders.csv")

df_rider = df_rider_raw.toDF(
    "rider_id",
    "first",
    "last",
    "address",
    "birthday",
    "account_start_date",
    "account_end_date",
    "is_member"
)

df_rider.write.format("delta").mode("overwrite").save("/delta/bronze/rider")

spark.read.format("delta").load("/delta/bronze/rider").show(3)

+----------+----------+------+--------------+
|payment_id|      date|amount|account_number|
+----------+----------+------+--------------+
|   1012544|2021-11-01|   9.0|         40136|
|   1012545|2021-12-01|   9.0|         40136|
|   1012546|2022-01-01|   9.0|         40136|
+----------+----------+------+--------------+
only showing top 3 rows
+------------+--------------------+-----------------+------------------+
|  station_id|                name|         latitude|         longitude|
+------------+--------------------+-----------------+------------------+
|         525|Glenwood Ave & To...|        42.012701|-87.66605799999999|
|KA1503000012|  Clark St & Lake St|41.88579466666667|-87.63110066666668|
|         637|Wood St & Chicago...|        41.895634|        -87.672069|
+------------+--------------------+-----------------+------------------+
only showing top 3 rows
+--------+-------+------+--------------------+----------+------------------+----------------+---------+
|rider_id|  fir

In [0]:
from pyspark.sql.functions import col, unix_timestamp

df_trip = spark.read.format("delta").load("/delta/bronze/trip")

df_trip_silver = df_trip \
    .withColumn("started_at", col("started_at").cast("timestamp")) \
    .withColumn("ended_at", col("ended_at").cast("timestamp")) \
    .withColumn(
        "trip_duration_minutes",
        (unix_timestamp("ended_at") - unix_timestamp("started_at")) / 60
    )

df_trip_silver.write.format("delta").mode("overwrite").save("/delta/silver/trip")

spark.read.format("delta") \
    .load("/delta/silver/trip") \
    .select("ride_id", "trip_duration_minutes") \
    .show(3)

+----------------+---------------------+
|         ride_id|trip_duration_minutes|
+----------------+---------------------+
|493E7531A95A143E|                 27.5|
|A3B7CB97068AA221|   30.533333333333335|
|E65F52226B594B89|    2.966666666666667|
+----------------+---------------------+
only showing top 3 rows


In [0]:
from pyspark.sql.functions import year, col

df_trip_silver = spark.read.format("delta").load("/delta/silver/trip").alias("t")
df_rider = spark.read.format("delta").load("/delta/bronze/rider").alias("r")

df_trip_silver = df_trip_silver.join(
    df_rider.select("rider_id", "birthday").alias("r"),
    col("t.rider_id") == col("r.rider_id"),
    "left"
).withColumn(
    "age_at_trip",
    year(col("t.started_at")) - year(col("r.birthday"))
)

# Seleciona apenas colunas do trip + nova métrica
df_trip_silver = df_trip_silver.select(
    "t.*",
    "age_at_trip"
)

df_trip_silver.write.format("delta").mode("overwrite").save("/delta/silver/trip")

spark.read.format("delta") \
    .load("/delta/silver/trip") \
    .select("rider_id", "age_at_trip") \
    .show(3)

+--------+-----------+
|rider_id|age_at_trip|
+--------+-----------+
|   71934|         38|
|   47854|         39|
|   70870|         34|
+--------+-----------+
only showing top 3 rows


In [0]:
from pyspark.sql.functions import to_date, year, month, quarter, dayofweek

df_trip_silver = spark.read.format("delta").load("/delta/silver/trip")

df_date = df_trip_silver.select(
    to_date("started_at").alias("date")
).distinct() \
.withColumn("year", year("date")) \
.withColumn("month", month("date")) \
.withColumn("quarter", quarter("date")) \
.withColumn("day_of_week", dayofweek("date"))

df_date.write.format("delta").mode("overwrite").save("/delta/gold/dim_date")

spark.read.format("delta") \
    .load("/delta/gold/dim_date") \
    .show(5)

+----------+----+-----+-------+-----------+
|      date|year|month|quarter|day_of_week|
+----------+----+-----+-------+-----------+
|2021-08-27|2021|    8|      3|          6|
|2021-09-14|2021|    9|      3|          3|
|2021-09-11|2021|    9|      3|          7|
|2021-08-21|2021|    8|      3|          7|
|2021-09-25|2021|    9|      3|          7|
+----------+----+-----+-------+-----------+
only showing top 5 rows


In [0]:
from pyspark.sql.functions import year

df_rider = spark.read.format("delta").load("/delta/bronze/rider")

df_rider_dim = df_rider.select(
    "rider_id",
    "is_member",
    "birthday",
    "account_start_date"
).withColumn(
    "age_at_account_start",
    year("account_start_date") - year("birthday")
)

df_rider_dim.write.format("delta").mode("overwrite").save("/delta/gold/dim_rider")

spark.read.format("delta") \
    .load("/delta/gold/dim_rider") \
    .select("rider_id", "is_member", "age_at_account_start") \
    .show(5)

+--------+---------+--------------------+
|rider_id|is_member|age_at_account_start|
+--------+---------+--------------------+
|   23344|     true|                  40|
|   23345|     true|                  30|
|   23346|     true|                  33|
|   23347|     true|                  46|
|   23348|     true|                  16|
+--------+---------+--------------------+
only showing top 5 rows


In [0]:
df_station = spark.read.format("delta").load("/delta/bronze/station")

df_station_dim = df_station.select(
    "station_id",
    "name",
    "latitude",
    "longitude"
)

df_station_dim.write.format("delta").mode("overwrite").save("/delta/gold/dim_station")

spark.read.format("delta") \
    .load("/delta/gold/dim_station") \
    .show(5)

+------------+--------------------+-----------------+------------------+
|  station_id|                name|         latitude|         longitude|
+------------+--------------------+-----------------+------------------+
|         525|Glenwood Ave & To...|        42.012701|-87.66605799999999|
|KA1503000012|  Clark St & Lake St|41.88579466666667|-87.63110066666668|
|         637|Wood St & Chicago...|        41.895634|        -87.672069|
|       13216|  State St & 33rd St|       41.8347335|       -87.6258275|
|       18003|Fairbanks St & Su...|41.89580766666667|-87.62025316666669|
+------------+--------------------+-----------------+------------------+
only showing top 5 rows


In [0]:
from pyspark.sql.functions import to_date

df_trip_silver = spark.read.format("delta").load("/delta/silver/trip")

df_fact_trip = df_trip_silver.select(
    "ride_id",
    "rider_id",
    "start_station_id",
    "end_station_id",
    to_date("started_at").alias("date"),
    "trip_duration_minutes",
    "age_at_trip"
)

df_fact_trip.write.format("delta").mode("overwrite").save("/delta/gold/fact_trip")

spark.read.format("delta") \
    .load("/delta/gold/fact_trip") \
    .select("ride_id", "trip_duration_minutes", "age_at_trip") \
    .show(5)

+----------------+---------------------+-----------+
|         ride_id|trip_duration_minutes|age_at_trip|
+----------------+---------------------+-----------+
|89E7AA6C29227EFF|    6.783333333333333|         38|
|0FEFDE2603568365|   19.516666666666666|         39|
|E6159D746B2DBB91|    8.866666666666667|         34|
|B32D3199F1C2E75B|    4.416666666666667|         19|
|83E463F23575F4BF|   15.233333333333333|         72|
+----------------+---------------------+-----------+
only showing top 5 rows


In [0]:
df_payment = spark.read.format("delta").load("/delta/bronze/payment")

df_fact_payment = df_payment.select(
    "payment_id",
    "account_number",
    "date",
    "amount"
)

df_fact_payment.write.format("delta").mode("overwrite").save("/delta/gold/fact_payment")

spark.read.format("delta") \
    .load("/delta/gold/fact_payment") \
    .select("payment_id", "amount") \
    .show(5)

+----------+------+
|payment_id|amount|
+----------+------+
|    512466|   9.0|
|    512467|   9.0|
|    512468|   9.0|
|    512469|   9.0|
|    512470|   9.0|
+----------+------+
only showing top 5 rows


In [0]:
spark.read.format("delta").load("/delta/gold/fact_trip").count()
spark.read.format("delta").load("/delta/gold/fact_payment").count()

1946607

In [0]:
spark.read.format("delta").load("/delta/gold/fact_trip").select("ride_id").distinct().count()

spark.read.format("delta").load("/delta/gold/fact_payment").select("payment_id").distinct().count()

1946607

Although Fact_Trip and Fact_Payment have similar row counts, they represent different business processes:

- Fact_Trip captures ride behavior (duration, usage)
- Fact_Payment captures financial transactions (amount spent)

This separation follows dimensional modeling best practices.